In [14]:
import importlib
import torch
import os
import pandas as pd
import src.dataset.dataset as dataset_module
import src.models.model as model_module
importlib.reload(dataset_module)
importlib.reload(model_module)
from src.dataset.dataset import GalaxyDataset
from torch.utils.data import DataLoader, dataset
from torchvision import transforms
from src.models.model import GalaxyClassifier_ResNet
import random

In [5]:
raw_transforms = transforms.Compose([
    transforms.CenterCrop(random.randint(170, 240)),  
    transforms.Resize(160),                            
    transforms.RandomCrop(128),                        
    transforms.RandomRotation(degrees = [90, 180]),
    transforms.RandomRotation(degrees = [0,270]),      
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1376, 0.1276, 0.1187],
                         std= [0.1728, 0.1586, 0.1488])    
])



In [15]:
labels = labels_df = pd.read_csv(r"D:\galaxy classification\data\preprocessed\labels\labeled_data_1.csv")
dataset = GalaxyDataset(img_dir= r"D:\galaxy classification\images", labels_df=labels_df, transform=raw_transforms)

2026-06-02 20:38:29.360 | WARNING  | src.dataset.dataset:__init__:35 - Dropped 37 rows with missing image files from the dataset.


In [7]:
sampler, class_weights = dataset.make_weighted_sampler()
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4, sampler=sampler)

In [8]:
sample_size = min(100, len(dataset))
if sample_size == 0:
    raise ValueError("No valid images were found in data/preprocessed/images. Run 'dvc pull' or point GalaxyDataset at a populated image directory.")

indices = random.sample(range(len(dataset)), sample_size)

subset_dataset = torch.utils.data.Subset(dataset, indices)
subset_sampler, _ = dataset.make_weighted_sampler(subset_dataset.indices)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GalaxyClassifier_ResNet(num_classes=5).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_mult = 2, eta_min=0.000001 ,T_0=10)

_,class_weights = dataset.make_weighted_sampler()

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1,weight = class_weights.to(device))


In [10]:
subset_dataloader = DataLoader(subset_dataset, batch_size=32, shuffle=False,sampler = subset_sampler, num_workers=4)

In [13]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
writer = SummaryWriter(log_dir = r"D:\galaxy classification\data\ml\tensorboard\experiment_1")

In [ ]:

global_step = 0
for epoch in range(10):
    model.train()

    pbar = tqdm(subset_dataloader, desc=f"Epoch {epoch+1}")

    for images, labels in subset_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)


        optimizer.step()
        scheduler.step(epoch+ global_step/len(subset_dataloader))
        
        global_step += 1

        writer.add_scalar("Loss/train", loss.item(), global_step)
        writer.add_scalar("LearningRate", optimizer.param_groups[0]['lr'], global_step)
        
        pbar.set_postfix({
            "loss": loss.item(),
            "lr": optimizer.param_groups[0]["lr"]
        })

    #val_loop()
    
     

Epoch [1/10], Loss: 1.9287
Epoch [1/10], Loss: 17.2242
Epoch [1/10], Loss: 68.0136
Epoch [1/10], Loss: 204.4397
Epoch [2/10], Loss: 78.8108
Epoch [2/10], Loss: 63.0421
Epoch [2/10], Loss: 46.3782
Epoch [2/10], Loss: 53.3022
Epoch [3/10], Loss: 45.5227
Epoch [3/10], Loss: 58.4275
Epoch [3/10], Loss: 58.7766
Epoch [3/10], Loss: 7.9528
Epoch [4/10], Loss: 12.3881
Epoch [4/10], Loss: 9.1449
Epoch [4/10], Loss: 7.3898
Epoch [4/10], Loss: 7.9487
Epoch [5/10], Loss: 18.0511
Epoch [5/10], Loss: 16.0031
Epoch [5/10], Loss: 12.5519
Epoch [5/10], Loss: 4.1269
Epoch [6/10], Loss: 12.5437
Epoch [6/10], Loss: 10.7491
Epoch [6/10], Loss: 13.2762
Epoch [6/10], Loss: 7.1126
Epoch [7/10], Loss: 8.2939
Epoch [7/10], Loss: 6.0501
Epoch [7/10], Loss: 7.3631
Epoch [7/10], Loss: 12.7929
Epoch [8/10], Loss: 7.5996
Epoch [8/10], Loss: 4.2917
Epoch [8/10], Loss: 5.7433
Epoch [8/10], Loss: 1.1312
Epoch [9/10], Loss: 4.6716
Epoch [9/10], Loss: 3.8064
Epoch [9/10], Loss: 3.8243
Epoch [9/10], Loss: 3.6158
Epoch [10

In [1]:
import pandas as pd

df = pd.read_csv(r"D:\galaxy classification\data\preprocessed\split\train_split.csv")
df_val = pd.read_csv(r"D:\galaxy classification\data\preprocessed\split\val_split.csv")

In [ ]:
df_sample = df.sample(n=200, random_state=42)
df_sample_val = df_val.sample(n=50, random_state=42)

In [4]:
df_sample.to_csv(r"D:\galaxy classification\data\preprocessed\split\train_sample.csv", index=False)
df_sample_val.to_csv(r"D:\galaxy classification\data\preprocessed\split\val_sample.csv", index=False)

In [1]:
from dotenv import load_dotenv
import os
import mlflow

d:\galaxy classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from src.utils.azure_utils import AzureBlobStorage
from src.utils.yaml_utils import load_yaml

In [7]:
file_config = r"D:\galaxy classification\configs\ml.yaml"
ml_yaml = load_yaml(file_config)


log_config = load_yaml(r"D:\galaxy classification\configs\core.yaml")

dev = ml_yaml["env"]["device"]


In [11]:
best_checkpoint_path = r"D:\galaxy classification\models\checkpoints\2026-06-11_17-34-30_Default\best.pt"

In [13]:
blob_path = f"checkpoints/1111111/best.pt"

In [8]:
blob_helper = AzureBlobStorage(container_name = "galaxy-experiments")

In [14]:

with open(best_checkpoint_path, "rb") as f:
    blob_url = blob_helper.upload_blob(blob_name=blob_path, data=f)
    




2026-06-11 19:34:59.988 | INFO     | src.utils.azure_utils:upload_blob:97 - Blob 'checkpoints/1111111/best.pt' uploaded successfully.


In [ ]:
#uploading logs to azure
log_system = AzureBlobStorage(container_name = log_config["logging"]["cloud"]["logs"]["container"])
log_blob_path = f"logs/{run_id}/training.log"
log_system.create_container()

with open(str(log_dir), "rb") as f:
    blob_log_url= log_system.upload_blob(blob_name=log_blob_path, data=f)
    
mlflow.set_tag("log_url", blob_log_url)

#uploading tensorboard logs to azure
with open(str(tensorboard_path), "rb") as f:
    tb_blob_path = f"tensorboard/{run_id}/"
    tb_blob_url = log_system.upload_blob(blob_name=tb_blob_path, data=f)
mlflow.set_tag("tensorboard_url", tb_blob_url)